# Embedding：从编号到向量

> 前两节完成了 Tokenizer：文本被切成 token，每个 token 分配了一个整数编号。但编号只是编号——7 和 3 之间没有大小关系，模型无法从编号判断两个词是否相似。
>
> 这一节引入 Embedding：把离散的 token ID 映射到一个连续的向量空间，让模型能够衡量词与词之间的距离。我们从分布式假设出发，手动构建共现矩阵，建立对词向量的直觉，最后组装出训练中实际使用的 Embedding 层。

Embedding 的核心作用是把离散的 token ID 映射到连续的向量空间。每个 token ID 不再是孤立的整数，而是对应一个固定长度的实数向量——比如 256 维或 768 维。这些向量一开始是随机初始化的，训练过程中模型通过反向传播不断调整它们。

训练完成之后，语义相近的词在向量空间中会靠得更近——cat 和 dog 的向量距离，会比 cat 和 algorithm 的向量距离近得多。这些向量不是人为指定的，而是从语料的统计规律中自然学到的。向量不会凭空拥有语义，让向量学会语义有两种做法：一是单独预训练词向量（如 Word2Vec），二是把 Embedding 作为模型的一部分端到端训练——这也是现代 LLM 采用的方式。这一节两种做法都会涉及。

In [1]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch 版本: {torch.__version__}")

PyTorch 版本: 2.11.0


## 本节要点

通过这一节的学习，以下问题应该能够回答：

1. 为什么 token ID 不能直接作为模型的输入？
2. 分布式假设是什么？它为什么是词向量的理论基础？
3. 共现矩阵是怎样从语料中构建出来的？
4. nn.Embedding 本质上在做什么？
5. 工业界训练 Embedding 时有哪些关键实践？

## 1. 从编号到向量

Tokenizer 把文本变成了整数序列，比如 [5, 1, 3]。但这些整数只是编号——5 和 1 之间没有大小关系，模型无法从编号判断两个词是否相似。

Embedding 要解决的问题就是：给每个 token ID 配一个固定长度的实数向量，让模型能在连续的向量空间里衡量词与词之间的关系。

这个想法可以从一个生活类比开始理解。用一个数字描述一个东西——比如只用身高——显然不够。168cm 的人有很多，他们各不相同。如果同时用多个特征来描述，情况就不一样了。

| | 尺寸 | 毛茸茸 | 与人亲近 | 独立性 |
|:---|:---|:---|:---|:---|
| cat | 2 | 8 | 6 | 9 |
| dog | 5 | 9 | 9 | 3 |
| algorithm | 0 | 0 | 0 | 0 |

每一行就是一组数值，可以写成一个向量：

```text
cat       → [2, 8, 6, 9]
dog       → [5, 9, 9, 3]
algorithm → [0, 0, 0, 0]
```

维度越多，描述越细。cat 和 dog 在每个维度上都比较接近，algorithm 和它们完全不一样。Embedding 做的事情本质上是相同的——只不过这些特征不是人工挑选的，而是模型在训练过程中自己学出来的。训练之后，语义相近的词在向量空间中会靠得更近。这就是 Embedding 的核心思想：用多个连续的数值联合描述一个 token，数值本身从数据中学来。

### 分布式表示：从颜色 RGB 到词向量

上面我们用 [尺寸, 毛茸茸, 与人亲近, 独立性] 四个维度描述了一个词，每个维度是一个连续的数值。用多个数值联合描述一个对象，而不是用一个孤立的 ID，是理解 Embedding 的关键。

先来看一个生活中的类比。

世界上存在各种各样的颜色。有的颜色被赋予了固定的名字，比如钴蓝（cobalt blue）或胭脂红（carmine）；颜色也可以用 RGB 三原色的强度来表示。前一种方式为每种颜色分配一个独立的名字——有多少种颜色，就需要多少个名字，而且名字之间没有数值关系。后一种方式将颜色表示为三维向量 (R, G, B)，比如 (201, 23, 30) 表示某种红色。

RGB 表示有三个好处：

1. **紧凑**：不管有多少种颜色，都只需要三个数值。
2. **精确**：(201, 23, 30) 比「深红色」更精确——不同人看到同一个 RGB 值能还原出同一种颜色。
3. **能表达相似性**：两个颜色的 RGB 向量越接近，颜色就越相似。(201, 23, 30) 和 (180, 20, 40) 都是红色系，而 (23, 180, 201) 是蓝色系——不需要知道颜色名字，直接算向量距离就能判断。

单词的表示面临类似的困境。给每个词分配一个独立 ID（像给颜色起名字），无法表达词与词之间的关系；但如果能把每个词表示为一个固定长度的实数向量（像用 RGB 表示颜色），词之间的相似性就可以通过向量距离来衡量。

在自然语言处理中，这种表示方式称为分布式表示（distributed representation）。它的核心特征是用密集向量（dense vector）表示单词——向量的各个元素大多是非零实数。比如一个三维分布式表示可能是 [0.21, −0.45, 0.83]。

这和 one-hot 编码形成鲜明对比。如果词表有 10000 个词，one-hot 是 10000 维的向量，其中 9999 个位置是 0，1 个位置是 1——稀疏、高维、无法表达词与词之间的关系。分布式表示则用低维密集向量（通常是 128、256 或 768 维），向量空间中的距离直接对应语义的远近。

接下来的问题是，如何从数据中自动学到这些向量。要回答这个问题，需要先理解分布式假设。

### 分布式假设：单词的含义由上下文决定

在自然语言处理的历史中，用向量表示单词的研究有很多。如果仔细审视这些研究，会发现几乎所有重要方法都基于一个简单的想法：**某个单词的含义由它周围的单词形成**。这个想法称为分布式假设（distributional hypothesis）。

分布式假设的含义很直接。单词本身没有含义，单词含义由它所在的上下文（语境）形成。含义相似的单词经常出现在相似的语境中。比如：

```text
I drink beer.    We drink wine.
I guzzle beer.   We guzzle wine.
```

drink 的附近常有饮料出现，guzzle 的附近也常有饮料出现——drink 和 guzzle 的上下文相似。基于这个观察，可以推断出 guzzle 和 drink 是近义词（guzzle 意为「大口喝」）。

这个例子展示了分布式假设的核心逻辑：不需要查字典，不需要语言学知识，仅凭单词和它周围词的共现模式，就能推断出单词之间的语义关系。

下面会频繁使用「上下文」这个词。上下文是指某个关注词周围的单词。上下文的大小（即周围单词的数量）称为窗口大小（window size）。窗口大小为 1，上下文包含左右各 1 个单词；窗口大小为 2，上下文包含左右各 2 个单词，以此类推。

```text
语料: You say goodbye and I say hello.

窗口大小为 2，关注词为 goodbye 时：
  You say goodbye and I say hello.
      ←──── 上下文 ────→

  goodbye 的上下文词 = {You, say, and, I}（左侧 2 个 + 右侧 2 个）
```

本节只处理左右单词数量相同、不考虑句子分隔符的情况。

### 共现矩阵：基于计数的最简实现

基于分布式假设，最直接的实现方法是统计周围单词的出现次数。对每个关注词，统计它的上下文中出现了哪些词、各出现了多少次，然后把所有词的统计结果汇总成一张矩阵。这种方法称为基于计数的方法（count-based method）。

下面用一个简单语料来手算一遍：

```text
You say goodbye and I say hello.
```

预处理后得到 7 个不同的词，ID 编号按首次出现顺序分配：`you=0, say=1, goodbye=2, and=3, i=4, hello=5, .=6`。

窗口大小设为 1，从单词 you 开始统计。you 的上下文窗口内只有一个词 say（窗口为 1，只看左右各 1 个词）：

```text
关注词: you
上下文 (window=1): [say]
→ 共现统计: {say: 1}
→ you 的向量: [0, 1, 0, 0, 0, 0, 0]（第 1 维=1 表示 say 出现了 1 次）
```

再统计 say。say 在语料中出现了两次（位置 1 和位置 5），每次的上下文分别是：

```text
第 1 次出现 (位置 1): 左=you, 右=goodbye → 统计 you:1, goodbye:1
第 2 次出现 (位置 5): 左=i, 右=hello    → 统计 i:1, hello:1
合并: {you:1, goodbye:1, i:1, hello:1}
→ say 的向量: [1, 0, 1, 0, 1, 1, 0]
```

对全部 7 个词都做同样的统计，结果按行堆叠，就得到共现矩阵（co-occurrence matrix）。矩阵的第 i 行就是词 i 的分布式表示——每一维表示对应单词在词 i 上下文中出现了多少次。

下面用代码实现这个统计过程。

In [ ]:
# === 从零实现 preprocess 和 create_co_matrix ===
import numpy as np

def preprocess(text):
    """文本 → 单词 ID 列表 + 词表映射"""
    text = text.lower()
    # 把句号从单词上拆下来（简化处理：只处理句号）
    text = text.replace('.', ' .')
    words = text.split()
    
    word_to_id = {}
    id_to_word = {}
    for word in words:
        if word not in word_to_id:
            new_id = len(word_to_id)
            word_to_id[word] = new_id
            id_to_word[new_id] = word
    
    corpus = [word_to_id[w] for w in words]
    return corpus, word_to_id, id_to_word


def create_co_matrix(corpus, vocab_size, window_size=1):
    """从语料库构建共现矩阵
    
    对语料中每个词，统计它窗口范围内的共现词频数。
    结果是一个 vocab_size × vocab_size 的矩阵，
    第 i 行第 j 列 = 词 j 出现在词 i 上下文中的次数。
    """
    corpus_size = len(corpus)
    co_matrix = np.zeros((vocab_size, vocab_size), dtype=np.int32)

    for idx, word_id in enumerate(corpus):
        for i in range(1, window_size + 1):
            left_idx = idx - i
            right_idx = idx + i

            # 左邻居（不越界则计入）
            if left_idx >= 0:
                left_word_id = corpus[left_idx]
                co_matrix[word_id, left_word_id] += 1

            # 右邻居（不越界则计入）
            if right_idx < corpus_size:
                right_word_id = corpus[right_idx]
                co_matrix[word_id, right_word_id] += 1

    return co_matrix


# ====== 用具体语料验证 ======
text = 'You say goodbye and I say hello.'
corpus, word_to_id, id_to_word = preprocess(text)

print(f"语料: {text}")
print(f"单词 ID 序列: {corpus}")
print(f"词表大小: {len(word_to_id)}")
print(f"ID → 词: {id_to_word}")

# 构建共现矩阵（窗口大小 = 1）
C = create_co_matrix(corpus, len(word_to_id), window_size=1)

print(f"\n共现矩阵形状: {C.shape} = [vocab_size={len(word_to_id)} × vocab_size={len(word_to_id)}]")
print(f"共现矩阵:\n{C}")

# 逐行观察每个词的向量
print("\n=== 每个词的共现向量（矩阵的每一行）===")
for word, idx in word_to_id.items():
    print(f"  '{word}' (ID={idx}): {C[idx].tolist()}")

print(f"\n关键观察：")
print(f"  you 的向量:    {C[word_to_id['you']].tolist()}  ← 只有 say=1，you 的上下文里只有 say")
print(f"  say 的向量:    {C[word_to_id['say']].tolist()}  ← you/goodbye/i/hello 各出现一次")
print(f"  goodbye 的向量: {C[word_to_id['goodbye']].tolist()}  ← say=1, and=1")
print(f"  → 每一行就是一个词的分布式表示，维度 = 词表大小")
print(f"  → 两个词的向量越相似 = 它们的上下文词分布越接近")


## 2. Embedding 查表

`nn.Embedding` 本质是一张矩阵，行数等于词表大小，列数等于向量维度：

```
矩阵形状: [vocab_size, embed_dim]

第 0 行 → token 0 的向量
第 1 行 → token 1 的向量
第 2 行 → token 2 的向量
...
```

给 Embedding 层一个 token ID，它就取出对应那一行。这些向量一开始是随机的，训练过程中模型会不断调整它们。训练之后，经常出现在相似上下文里的词，向量会靠得更近。

**Embedding 是怎么训练出来的**

向量不会凭空拥有语义——它需要从数据中学。业界有两种主要做法：

第一种是**预训练词向量**，以 Word2Vec 和 GloVe 为代表。思路是单独训练 Embedding，然后把它当作固定输入喂给下游模型。比如 Word2Vec 的 Skip-gram 方法：给定一个词，预测它周围可能出现哪些词。通过这个任务，模型被迫学会"哪些词经常出现在相似语境中"，从而让这些词的向量靠近。

第二种是**端到端训练**，也是现代 LLM 采用的方式。Embedding 矩阵作为模型的一部分参数，和 Transformer 的其他层一起，通过反向传播统一更新：

```text
初始化: Embedding 矩阵 E ← 随机值

训练循环:
  for (input_ids, target_ids) in data:
      vectors = E[input_ids]                    # 查表取向量
      logits  = Transformer(vectors)            # 模型前向传播
      loss    = CrossEntropy(logits, targets)   # 计算损失
      loss.backward()                           # 反向传播
      E       = E - lr × E.grad                 # 更新 Embedding 矩阵
```

经过足够多的训练步数，Embedding 矩阵中的每一行就会逐渐学出有意义的向量表示。后面实现 Mini-GPT 时会看到完整的训练流程。

In [ ]:
# 模拟一个 mini 词表，Embedding = vocab_size × embed_dim 的矩阵
vocab = ["the", "cat", "sat", "on", "mat", "dog", "log"]
vocab_size = len(vocab)
embed_dim = 4

embedding = nn.Embedding(vocab_size, embed_dim)

print(f"词表大小: {vocab_size}, Embedding 维度: {embed_dim}")
print(f"Embedding 权重形状: {embedding.weight.shape}  ← 就是一个 {vocab_size}×{embed_dim} 矩阵")
print(f"\n前 3 行初始值（随机）:\n{embedding.weight[:3]}")

In [ ]:
# 查表：给一组 token ID，取出对应的向量
sentence_ids = torch.tensor([0, 1, 2, 3, 0, 4])  # "the cat sat on the mat"
vectors = embedding(sentence_ids)                  # 查表 → [6, 4]

print(f"token IDs: {sentence_ids.tolist()}  →  {[vocab[i] for i in sentence_ids.tolist()]}")
print(f"输出形状: {vectors.shape}  ← [{len(sentence_ids)} 个 token, 每个 {embed_dim} 维]")
print()

# 逐个看
for i, (tid, vec) in enumerate(zip(sentence_ids.tolist(), vectors)):
    print(f"  位置 {i}: '{vocab[tid]}' (ID={tid}) → {vec.tolist()}")

# 关键观察：位置 0 和 4 都是 'the'，向量完全相同
# → 同一个 token 不管出现在哪个位置，查出的向量都一样
print(f"\n关键观察：位置 0 和 4 都是 token 'the'，查出的向量完全相同")
print(f"→ 同一个 token 不管出现在哪个位置，Embedding 查出来的向量都一样")
print(f"→ 模型还无法区分词的顺序——这是下一节位置编码要解决的问题")

## 3. 工业界的 Embedding 训练实践

前面的 nn.Embedding 查表把概念讲清楚了。但在真实的大语言模型训练中，还有几个工程决策直接影响参数效率和训练稳定性。下面逐一来看。

**权重共享（Weight Tying）**

模型的开头和结尾各有一个大矩阵：输入端的 Embedding 把 token ID 变成向量，输出端的 lm_head 把向量变回词表大小的 logits。两个矩阵形状相同，都是 [vocab_size, d_model]——前者查表取向量，后者把向量投影回词表空间做概率预测。

既然形状相同，能不能共用同一组参数？可以。这种做法称为权重共享（weight tying）：输入 Embedding 和输出投影指向同一块内存，查表和投影用的是同一张矩阵。

以 LLaMA 7B 为例：vocab_size=32000，d_model=4096，不共享要多约 1.3 亿参数。GPT-2、GPT-3、LLaMA、DeepSeek 等都采用了这个技巧。

**Embedding 不做权重衰减**

L2 正则化（weight decay）限制权重的绝对值，防止过拟合。但 Embedding 的每一行代表一个 token 的语义向量——它需要在训练中自由移动到合适的位置。施加 L2 惩罚等于把所有向量往原点拽，干扰语义学习。

业界惯例：Embedding、LayerNorm 的 weight/bias、所有 bias 项都不参与 weight decay。训练时把参数分成两组，一组正常衰减，一组衰减系数为零。

**初始化：更小的标准差**

nn.Embedding 默认用 N(0,1) 初始化。实践中通常用更小的标准差——例如 N(0, 1/√d_model)——避免初始值过大，导致训练早期的梯度不稳定。

**混合精度下的 Embedding**

用 FP16/BF16 训练时，Embedding 矩阵通常保留一份 FP32 副本（master weights）。前向传播转成低精度以加速计算，反向更新时在 FP32 下累积梯度。这是为了避免 FP16 动态范围不足导致的梯度下溢。

下面用代码把这几个实践串起来，模拟一个真实训练循环中 Embedding 的配置方式。

In [ ]:
# === 工业界 Embedding 训练实践：权重共享 + 参数分组 + 初始化 ===
import math

# ---------- 1. 权重共享（Weight Tying）----------
# 输入 Embedding 和输出 lm_head 共享权重——这是 GPT/LLaMA 的标准做法
vocab_size, d_model = 10000, 512

class GPTStyleModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)   # token → vector
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)  # vector → token probs
        # ★ 关键：让 lm_head 复用 wte 的权重矩阵
        self.lm_head.weight = self.wte.weight

    def forward(self, input_ids):
        x = self.wte(input_ids)
        return self.lm_head(x)

model = GPTStyleModel(vocab_size, d_model)

# 验证：两个 weight 指向同一块内存
print("=" * 50)
print("1. 权重共享验证")
print(f"   wte.weight 内存地址:       {model.wte.weight.data_ptr()}")
print(f"   lm_head.weight 内存地址:   {model.lm_head.weight.data_ptr()}")
print(f"   是同一块内存:              {model.wte.weight.data_ptr() == model.lm_head.weight.data_ptr()}")
print(f"   节省参数量:                {vocab_size * d_model:,} → {vocab_size * d_model * 4 / 1e6:.1f} MB (fp32)")

# 反向传播验证：同一块内存上的梯度正确累加
input_ids = torch.randint(0, vocab_size, (2, 16))
logits = model(input_ids)
logits.mean().backward()
print(f"   两个 grad 指向同一内存:    {model.wte.weight.grad.data_ptr() == model.lm_head.weight.grad.data_ptr()}")

# ---------- 2. 不共享 vs 共享：参数量对比 ----------
class NoTieModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # 没有 weight tying，两个矩阵各自独立

    def forward(self, input_ids):
        return self.lm_head(self.wte(input_ids))

tie_params = sum(p.numel() for p in GPTStyleModel(vocab_size, d_model).parameters())
no_tie_params = sum(p.numel() for p in NoTieModel(vocab_size, d_model).parameters())
print(f"\n2. 参数量对比 (vocab={vocab_size:,}, d_model={d_model}):")
print(f"   共享权重: {tie_params:,}")
print(f"   独立权重: {no_tie_params:,}")
print(f"   节省:     {no_tie_params - tie_params:,} ({100 * (no_tie_params - tie_params) / no_tie_params:.1f}%)")

# 类比真实模型尺寸
for name, vs, dm in [("GPT-2 small", 50257, 768), ("LLaMA 7B", 32000, 4096)]:
    saved = vs * dm
    print(f"   {name}: vocab={vs:,}, d_model={dm} → 节省 {saved:,} 参数 ({saved * 4 / 1e6:.1f} MB)")

# ---------- 3. Embedding 不做 weight decay ----------
# LLaMA / GPT 训练的标准参数分组：某些参数不做正则化
def make_param_groups(model, weight_decay=0.1):
    """把参数分成两组：一组做 decay，一组不做"""
    decay_params, no_decay_params = [], []
    no_decay_keywords = ['wte', 'norm', 'bias']  # Embedding / Norm / Bias 不做衰减
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(kw in name.lower() for kw in no_decay_keywords):
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]

model2 = GPTStyleModel(vocab_size, d_model)
param_groups = make_param_groups(model2, weight_decay=0.1)

decay_count = sum(p.numel() for p in param_groups[0]['params'])
no_decay_count = sum(p.numel() for p in param_groups[1]['params'])
print(f"\n3. 参数分组（weight_decay=0.1）:")
print(f"   做衰减的参数:   {decay_count:,}")
print(f"   不做衰减的参数: {no_decay_count:,}  ← Embedding 在这里")
print(f"   Embedding 在 no_decay 组: {any('wte' in n for n, _ in model2.named_parameters())}")

optimizer = torch.optim.AdamW(param_groups, lr=1e-3)
for i, pg in enumerate(optimizer.param_groups):
    wd = pg['weight_decay']
    count = sum(p.numel() for p in pg['params'])
    print(f"   Group {i}: {count:,} params, weight_decay={wd}")

# ---------- 4. Embedding 初始化 ----------
print(f"\n4. 初始化对比:")
torch.manual_seed(42)
default_emb = nn.Embedding(100, 16)
custom_emb = nn.Embedding(100, 16)
nn.init.normal_(custom_emb.weight, mean=0.0, std=1.0 / math.sqrt(d_model))

print(f"   nn.Embedding 默认 N(0,1):       std = {default_emb.weight.std().item():.4f}")
print(f"   自定义 N(0, 1/√d_model):         std = {custom_emb.weight.std().item():.4f}")
print(f"   目标 1/√{d_model} = {1.0 / math.sqrt(d_model):.4f}")
print(f"   → 更小的初始值 → 训练早期梯度更稳定")

# 模拟一个 mini-step：走一遍完整的训练步骤
print(f"\n5. 模拟一次训练 step（含梯度裁剪）:")
model3 = GPTStyleModel(vocab_size, d_model)
optimizer = torch.optim.AdamW(
    make_param_groups(model3, weight_decay=0.1), lr=1e-3
)

batch = torch.randint(0, vocab_size, (4, 32))   # 随机 batch
targets = torch.randint(0, vocab_size, (4, 32))  # 随机 targets

logits = model3(batch)                            # 前向
loss = nn.functional.cross_entropy(
    logits.view(-1, vocab_size), targets.view(-1)
)
loss.backward()                                   # 反向
torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0)  # 梯度裁剪
optimizer.step()                                  # 更新参数
optimizer.zero_grad()

print(f"   loss: {loss.item():.4f}")
print(f"   wte.weight 已更新: {model3.wte.weight.grad is None} → 梯度已清零")
print(f"   → 完整流程：forward → loss → backward → clip → step → zero_grad")


## 小结

这一节所学的内容：

- Token ID 只是编号，不能直接作为模型的数值输入
- Embedding 是一张 [vocab_size, d_model] 的矩阵，给定 ID 就取出对应行的向量
- 分布式假设：一个词的含义由它周围的词决定，含义相似的词出现在相似的语境中
- 共现矩阵是基于计数的最简分布式表示——统计每个词的上下文词频
- nn.Embedding 是可学习的查表，训练后语义相近的词在向量空间中距离更近
- 工业界实践：权重共享节省参数、Embedding 不做 weight decay、更小的初始化标准差

同一个 token 不管出现在哪个位置，查出的向量都相同——模型还需要知道每个 token 在句子中的位置。下一节引入位置编码来解决这个问题。

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：Embedding 查表**

Token ID 本身没有语义，Embedding 会把 ID 查成向量。

小提示：`embedding_table[token_ids]` 可以一次取多行。

In [ ]:
# 作业 1：Embedding 查表填空
import torch

embedding_table = torch.tensor([
    [1.0, 0.0],  # token 0
    [0.0, 1.0],  # token 1
    [1.0, 1.0],  # token 2
])
token_ids = torch.tensor([2, 0, 1])

# TODO：把下面三引号里的内容替换成你的代码
vectors = """在这里根据 token_ids 从 embedding_table 里取出对应向量"""

assert not isinstance(vectors, str), "请先替换三引号里的占位内容"
expected = torch.tensor([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
assert torch.equal(vectors, expected), vectors
print("✅ 作业 1 通过：你记住了 Embedding 的核心就是查表")

## 参考资料

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — Transformer 原始论文，Embedding 乘以 √d_model 的惯例来自此文
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — 对原始论文的逐行实现，`Embeddings` 类中可以看到 `self.lut(x) * math.sqrt(self.d_model)` 的写法
- Mikolov et al., [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781), 2013 — Word2Vec，分布式表示的经典工作